# Data summary

Read in all the datasets and report various metrics.


In [19]:
import pandas as pd

# Read in the folder
data_dir = '../manuscript/EnzEngDB_V2/experiments/'
meta_data = pd.read_excel('../manuscript/EnzEngDB_V2/meta_data.xlsx')

# QC to ensure everything is correct

1. Ensure that all experiments have a #PARENT# enzyme sequence
2. Ensure all enzymes have a smiles string (for a product at least)
3. Ensure all enzymes have a fitness value (or set to be trace 0.001)
4. Plot all smiles strings to just get an overview of the diversity of the data

In [20]:
# Read in each of the experiments and write in how many entries are in there
num_proteins, num_rows, num_fit, num_rxns = [], [], [], []
df = pd.DataFrame(meta_data)
for exp in df['experiment_id'].values:
    exp_df = pd.read_csv(f'../manuscript/EnzEngDB_V2/experiments/{exp}.csv')
    exp_df = exp_df.dropna(subset='fitness_value')
    unique_proteins = len(set(list(exp_df['aa_sequence'].values)))
    unique_reactions = len(set(list(exp_df['smiles_string'].values)))
    unique_fitness = len(set(list(exp_df['fitness_value'].values)))
    # Check it has a parent value and this matches the parent value in the metadata file
    
    num_proteins.append(unique_proteins)
    num_rows.append(len(exp_df))
    num_fit.append(unique_fitness)
    num_rxns.append(unique_reactions)
    all_df = pd.concat([all_df, exp_df])

df['num_proteins'] = num_proteins
df['num_rows'] = num_rows
df['num_fitness_entries'] = num_fit
df['num_rxns'] = num_rxns

# Here we want to just report some statistics about the dataset

1. Number of data points in the GOLD standard dataset
2. Number of data points in the LLM dataset
3. Number of data points in the large datasets
4. Sequence length distribution
5. Pairwise similarity of the reactions

In [21]:
import numpy as np
from rdkit import Chem
from rdkit.Chem import AllChem
from rdkit import DataStructs

# 1-3. Count data points by dataset type using the 'prefix' column
gold_count = df[df['prefix'] == 'ARNLD']['num_rows'].sum()
llm_count = df[df['prefix'] == 'LLMDB']['num_rows'].sum()
large_count = df[df['prefix'] == 'DMSDB']['num_rows'].sum()

print(f"GOLD standard dataset (ARNLD): {gold_count} data points")
print(f"LLM dataset (LLMDB): {llm_count} data points")
print(f"Large datasets (DMSDB): {large_count} data points")
print(f"Total: {gold_count + llm_count + large_count} data points")

# 4. Sequence length distribution from all CSVs
all_sequences = []
for exp in df['experiment_id'].values:
    exp_df = pd.read_csv(f'{data_dir}{exp}.csv')
    all_sequences.extend(exp_df['aa_sequence'].dropna().values)

seq_lengths = [len(seq) for seq in all_sequences]
print(f"\nSequence length statistics:")
print(f"  Mean: {np.mean(seq_lengths):.1f}")
print(f"  Median: {np.median(seq_lengths):.1f}")
print(f"  Min: {np.min(seq_lengths)}")
print(f"  Max: {np.max(seq_lengths)}")
print(f"  Std: {np.std(seq_lengths):.1f}")

# 5. Pairwise similarity of reactions using RDKit
# Collect all unique reaction SMILES strings (check both column names)
all_reaction_smiles = []
for exp in df['experiment_id'].values:
    exp_df = pd.read_csv(f'{data_dir}{exp}.csv')
    # Some files use 'smiles_reaction', others use 'reaction_smiles'
    if 'smiles_reaction' in exp_df.columns:
        all_reaction_smiles.extend(exp_df['smiles_reaction'].dropna().unique())
    elif 'reaction_smiles' in exp_df.columns:
        all_reaction_smiles.extend(exp_df['reaction_smiles'].dropna().unique())

unique_reactions = list(set(all_reaction_smiles))
print(f"\nTotal unique reactions: {len(unique_reactions)}")

# Calculate Morgan fingerprints and pairwise Tanimoto similarities for reactions
# Use reaction fingerprints
from rdkit.Chem import rdChemReactions
fps = []
for rxn_smi in unique_reactions:
    try:
        rxn = rdChemReactions.ReactionFromSmarts(rxn_smi, useSmiles=True)
        if rxn:
            fp = rdChemReactions.CreateStructuralFingerprintForReaction(rxn)
            fps.append(fp)
    except:
        # If reaction parsing fails, skip it
        pass

# Calculate pairwise similarities
similarities = []
for i in range(len(fps)):
    for j in range(i+1, len(fps)):
        sim = DataStructs.TanimotoSimilarity(fps[i], fps[j])
        similarities.append(sim)

if similarities:
    print(f"\nPairwise reaction similarity (Tanimoto):")
    print(f"  Mean: {np.mean(similarities):.3f}")
    print(f"  Median: {np.median(similarities):.3f}")
    print(f"  Min: {np.min(similarities):.3f}")
    print(f"  Max: {np.max(similarities):.3f}")
    print(f"  Std: {np.std(similarities):.3f}")

GOLD standard dataset (ARNLD): 6415 data points
LLM dataset (LLMDB): 2415 data points
Large datasets (DMSDB): 449745 data points
Total: 458575 data points

Sequence length statistics:
  Mean: 335.2
  Median: 390.0
  Min: 5
  Max: 1058
  Std: 109.6

Total unique reactions: 1808


[11:33:14] SMILES Parse Error: syntax error while parsing: [N2]CC#N
[11:33:14] SMILES Parse Error: check for mistakes around position 3:
[11:33:14] [N2]CC#N
[11:33:14] ~~^
[11:33:14] SMILES Parse Error: Failed parsing SMILES '[N2]CC#N' for input: '[N2]CC#N'
[11:33:14] SMILES Parse Error: syntax error while parsing: [N2]CC#N
[11:33:14] SMILES Parse Error: check for mistakes around position 3:
[11:33:14] [N2]CC#N
[11:33:14] ~~^
[11:33:14] SMILES Parse Error: Failed parsing SMILES '[N2]CC#N' for input: '[N2]CC#N'
[11:33:14] SMILES Parse Error: syntax error while parsing: [N2]CC#N
[11:33:14] SMILES Parse Error: check for mistakes around position 3:
[11:33:14] [N2]CC#N
[11:33:14] ~~^
[11:33:14] SMILES Parse Error: Failed parsing SMILES '[N2]CC#N' for input: '[N2]CC#N'
[11:33:14] SMILES Parse Error: syntax error while parsing: [N2]CC#N
[11:33:14] SMILES Parse Error: check for mistakes around position 3:
[11:33:14] [N2]CC#N
[11:33:14] ~~^
[11:33:14] SMILES Parse Error: Failed parsing SMILES '


Pairwise reaction similarity (Tanimoto):
  Mean: 0.431
  Median: 0.417
  Min: 0.013
  Max: 1.000
  Std: 0.139
